# GroupDNA — WhatsApp Group Chat Analyzer

## Minor Project — The Unlox Academy

**Name:** Syeda Inshiraah  
**USN / Roll Number:** 4MH23CS167  
**Batch:** Data Analytics  
**Date:** 21 September 2026

---

### Project Objective

GroupDNA is a WhatsApp group chat analytics tool built using
Python fundamentals and NumPy.

The project analyzes:
- Group activity
- Message frequency
- Most active days and hours
- Activity patterns using a NumPy heatmap
- Frequently used words
- Response patterns
- Silent streaks
- Personality archetypes

# Feature 1 — Chat Parser

This section reads the WhatsApp export file, identifies valid
message lines, extracts timestamps, sender names and message text,
and handles system messages, media-omitted messages and deleted
messages.

In [13]:
# AI tools were used as a learning aid during development.

from datetime import datetime, timedelta

file_name = "hostel_bois.txt"

messages = []
system_messages = []
media_count = 0
deleted_count = 0

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

current_message = None

for line in lines:
    line = line.strip()

    if line == "":
        continue

    if " - " in line:
        parts = line.split(" - ", 1)
        date_part = parts[0]
        message_part = parts[1]

        date_time = None

        formats = [
            "%d/%m/%y, %H:%M",
            "%d/%m/%Y, %H:%M",
            "%d/%m/%y, %I:%M %p",
            "%d/%m/%Y, %I:%M %p"
        ]

        for fmt in formats:
            try:
                date_time = datetime.strptime(date_part, fmt)
                break
            except ValueError:
                pass

        if date_time is not None:

            if current_message is not None:
                messages.append(current_message)

            if ": " in message_part:
                sender, text = message_part.split(": ", 1)

                current_message = {
                    "sender": sender,
                    "datetime": date_time,
                    "text": text
                }
            else:
                system_messages.append(message_part)
                current_message = None

            continue

    if current_message is not None:
        current_message["text"] += " " + line

if current_message is not None:
    messages.append(current_message)

for message in messages:
    if message["text"] == "<Media omitted>":
        media_count += 1

    if message["text"] == "This message was deleted":
        deleted_count += 1

print("Total real messages:", len(messages))
print("System messages:", len(system_messages))
print("Media messages:", media_count)
print("Deleted messages:", deleted_count)

Total real messages: 3174
System messages: 4
Media messages: 32
Deleted messages: 15


##2. Group Overview


In [14]:
participants = set()

for message in messages:
    participants.add(message["sender"])

message_counts = {}

for participant in participants:
    message_counts[participant] = 0

for message in messages:
    message_counts[message["sender"]] += 1

dates = []

for message in messages:
    dates.append(message["datetime"])

start_date = min(dates)
end_date = max(dates)

print("╔══════════════════════════════════════╗")
print("║           GROUP OVERVIEW             ║")
print("╚══════════════════════════════════════╝")

print("Participants :", len(participants))
print("Messages     :", len(messages))
print("Media        :", media_count)
print("Deleted      :", deleted_count)
print("Start date   :", start_date.strftime("%d %B %Y"))
print("End date     :", end_date.strftime("%d %B %Y"))

print("\nMessages per participant:")

for participant in sorted(message_counts, key=message_counts.get, reverse=True):
    print(participant, ":", message_counts[participant])

╔══════════════════════════════════════╗
║           GROUP OVERVIEW             ║
╚══════════════════════════════════════╝
Participants : 6
Messages     : 3174
Media        : 32
Deleted      : 15
Start date   : 01 April 2024
End date     : 30 May 2024

Messages per participant:
Rahul : 953
Priya : 718
Neha : 635
Aman : 490
Karan : 354
Vikas : 24


##3. Most Active Day and Hour

In [15]:
day_counts = {}
hour_counts = {}

for message in messages:
    day = message["datetime"].strftime("%A")
    hour = message["datetime"].hour

    if day not in day_counts:
        day_counts[day] = 0

    if hour not in hour_counts:
        hour_counts[hour] = 0

    day_counts[day] += 1
    hour_counts[hour] += 1

most_active_day = max(day_counts, key=day_counts.get)
most_active_hour = max(hour_counts, key=hour_counts.get)

print("Most Active Day :", most_active_day)
print("Messages on Day :", day_counts[most_active_day])
print("Most Active Hour:", f"{most_active_hour:02d}:00")
print("Messages in Hour:", hour_counts[most_active_hour])

print("\nDay-wise Activity:")

for day in sorted(day_counts, key=day_counts.get, reverse=True):
    print(day, ":", day_counts[day])

print("\nHour-wise Activity:")

for hour in sorted(hour_counts):
    print(f"{hour:02d}:00 - {hour:02d}:59 :", hour_counts[hour])

Most Active Day : Wednesday
Messages on Day : 483
Most Active Hour: 18:00
Messages in Hour: 248

Day-wise Activity:
Wednesday : 483
Monday : 482
Thursday : 475
Saturday : 459
Sunday : 446
Tuesday : 426
Friday : 403

Hour-wise Activity:
00:00 - 00:59 : 57
01:00 - 01:59 : 82
02:00 - 02:59 : 83
03:00 - 03:59 : 77
04:00 - 04:59 : 110
05:00 - 05:59 : 29
06:00 - 06:59 : 33
07:00 - 07:59 : 55
08:00 - 08:59 : 122
09:00 - 09:59 : 151
10:00 - 10:59 : 160
11:00 - 11:59 : 114
12:00 - 12:59 : 193
13:00 - 13:59 : 159
14:00 - 14:59 : 162
15:00 - 15:59 : 131
16:00 - 16:59 : 189
17:00 - 17:59 : 173
18:00 - 18:59 : 248
19:00 - 19:59 : 228
20:00 - 20:59 : 166
21:00 - 21:59 : 177
22:00 - 22:59 : 116
23:00 - 23:59 : 159


##4. Activity Heatmap

In [16]:
import numpy as np

heatmap = np.zeros((6, 24), dtype=int)

participant_rows = {
    "Rahul": 0,
    "Priya": 1,
    "Aman": 2,
    "Karan": 3,
    "Neha": 4,
    "Vikas": 5
}

for message in messages:
    sender = message["sender"]
    hour = message["datetime"].hour

    if sender in participant_rows:
        heatmap[participant_rows[sender]][hour] += 1

print("Activity Heatmap")
print()

print("Participant  ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()

for participant, row in participant_rows.items():
    print(f"{participant:<12}", end="")

    for hour in range(24):
        value = heatmap[row][hour]

        if value == 0:
            block = " "
        elif value < 10:
            block = "░"
        elif value < 25:
            block = "▒"
        elif value < 50:
            block = "▓"
        else:
            block = "█"

        print(f"{block}  ", end="")

    print()

print("\nHeatmap Matrix Shape:", heatmap.shape)

Activity Heatmap

Participant  00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Rahul       ░  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▒  ▓  ▒  █  ▓  ▓  █  █  ▓  █  █  ▓  █  █  █  
Priya                         ▒  ▒  ▓  █  █  █  █  ▓  ▓  ▓  ▓  ▓  ▓  █  ▓  ▓  ▒  ░  
Aman        █  █  █  █  █                             ▒  ▒  ▒  ░  ▒  ░  ▒  ▒     █  
Karan                            ░  ▒  ▒  ▒  ▒  ▓  ▓  ▓  ▓  ▓  ▓  ▓  ▓  ▒  ▒  ░  ░  
Neha                       ▒  ░  ▒  ▓  █  █  ▒  ▓  ▓  ▓  ▒  ▓  ▓  █  █  ▓  ▓  ▓  ▓  
Vikas                            ░  ░  ░  ░     ░  ░     ░  ░  ░  ░  ░  ░  ░  ░  ░  

Heatmap Matrix Shape: (6, 24)


##5. Top Words

In [17]:
word_counts = {}

stop_words = {
    "the", "is", "a", "an", "and", "to", "of", "in", "for",
    "on", "at", "it", "this", "that", "i", "you", "we", "he",
    "she", "they", "was", "are", "be", "with", "but", "or",
    "so", "my", "me", "your", "our", "have", "has", "had",
    "not", "do", "did", "will", "can", "just", "from"
}

for message in messages:
    words = message["text"].lower().split()

    for word in words:
        word = word.strip(".,!?;:\"'()[]{}")

        if word != "" and word not in stop_words:
            if word not in word_counts:
                word_counts[word] = 0

            word_counts[word] += 1

top_words = sorted(
    word_counts,
    key=word_counts.get,
    reverse=True
)[:20]

print("Top 20 Words")
print()

for word in top_words:
    print(f"{word:<20} {word_counts[word]}")

Top 20 Words

how                  321
guys                 318
about                274
hai                  268
am                   260
today                257
his                  217
which                202
everyone             187
telling              179
up                   172
bhai                 160
one                  157
started              150
no                   146
scene                145
entire               145
please               141
anyone               139
yaar                 139


##6. Response Speed and Slient Streaks

In [18]:
sorted_messages = sorted(messages, key=lambda x: x["datetime"])

response_times = {}
silent_days = {}

for participant in participants:
    response_times[participant] = []
    silent_days[participant] = set()

for i in range(1, len(sorted_messages)):
    previous = sorted_messages[i - 1]
    current = sorted_messages[i]

    if previous["sender"] != current["sender"]:
        difference = current["datetime"] - previous["datetime"]

        if difference.total_seconds() <= 3600:
            response_times[current["sender"]].append(
                difference.total_seconds() / 60
            )

for participant in participants:
    active_dates = set()

    for message in messages:
        if message["sender"] == participant:
            active_dates.add(message["datetime"].date())

    current_date = start_date.date()

    while current_date <= end_date.date():
        if current_date not in active_dates:
            silent_days[participant].add(current_date)

        current_date = current_date + timedelta(days=1)

print("Average Response Time")
print()

for participant in sorted(response_times):
    times = response_times[participant]

    if len(times) > 0:
        average = sum(times) / len(times)
        print(f"{participant:<10} {average:.2f} minutes")
    else:
        print(f"{participant:<10} No responses")

print("\nSilent Days")

for participant in sorted(silent_days):
    print(f"{participant:<10} {len(silent_days[participant])} days")

Average Response Time

Aman       23.21 minutes
Karan      21.68 minutes
Neha       19.32 minutes
Priya      20.20 minutes
Rahul      19.26 minutes
Vikas      24.65 minutes

Silent Days
Aman       0 days
Karan      0 days
Neha       0 days
Priya      0 days
Rahul      0 days
Vikas      44 days


##7.Personality Archetpyes

In [19]:
archetypes = [
    "THE SPAMMER",
    "THE GROUP MOM",
    "THE NIGHT OWL",
    "THE STORYTELLER",
    "THE DRAMA QUEEN",
    "THE GHOST",
    "THE COMEDIAN",
    "THE QUESTION MASTER"
]

caring_words = [
    "okay", "safe", "eat", "sleep", "take care",
    "are you", "please", "reminder", "drink water",
    "don't forget"
]

comedian_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

scores = {}

for participant in participants:
    scores[participant] = {}

    participant_messages = []

    for message in messages:
        if message["sender"] == participant:
            participant_messages.append(message)

    total = len(participant_messages)

    burst_lengths = []
    current_burst = 0
    previous_sender = ""

    for message in sorted_messages:
        if message["sender"] == participant:
            if previous_sender == participant:
                current_burst += 1
            else:
                if current_burst > 0:
                    burst_lengths.append(current_burst)

                current_burst = 1

            previous_sender = participant
        else:
            if current_burst > 0:
                burst_lengths.append(current_burst)

            current_burst = 0
            previous_sender = message["sender"]

    if current_burst > 0:
        burst_lengths.append(current_burst)

    if len(burst_lengths) > 0:
        average_burst = sum(burst_lengths) / len(burst_lengths)
    else:
        average_burst = 0

    caring_count = 0

    for message in participant_messages:
        text = message["text"].lower()

        for word in caring_words:
            if word in text:
                caring_count += 1

    night_count = 0

    for message in participant_messages:
        hour = message["datetime"].hour

        if hour >= 23 or hour <= 4:
            night_count += 1

    night_percentage = (night_count / total) * 100

    total_words = 0

    for message in participant_messages:
        total_words += len(message["text"].split())

    average_words = total_words / total

    drama_count = 0

    for message in participant_messages:
        text = message["text"]

        uppercase_count = 0
        letter_count = 0

        for character in text:
            if character.isalpha():
                letter_count += 1

                if character.isupper():
                    uppercase_count += 1

        uppercase_percentage = 0

        if letter_count > 0:
            uppercase_percentage = (uppercase_count / letter_count) * 100

        if (uppercase_percentage == 100 and len(text) >= 3) or "!!" in text:
            drama_count += 1

    drama_percentage = (drama_count / total) * 100

    silent_percentage = (len(silent_days[participant]) / 60) * 100

    comedian_count = 0

    for message in participant_messages:
        words = message["text"].lower().split()

        for word in words:
            word = word.strip(".,!?;:\"'()[]{}")

            if word in comedian_words:
                comedian_count += 1

    comedian_percentage = (comedian_count / total) * 100

    question_count = 0

    for message in participant_messages:
        if message["text"].strip().endswith("?"):
            question_count += 1

    question_percentage = (question_count / total) * 100

    scores[participant]["THE SPAMMER"] = 1 if average_burst > 3 else 0
    scores[participant]["THE GROUP MOM"] = 0
    scores[participant]["THE NIGHT OWL"] = 1 if night_percentage > 60 else 0
    scores[participant]["THE STORYTELLER"] = 1 if average_words > 30 else 0
    scores[participant]["THE DRAMA QUEEN"] = 1 if drama_percentage > 30 else 0
    scores[participant]["THE GHOST"] = 1 if silent_percentage > 60 else 0
    scores[participant]["THE COMEDIAN"] = 0
    scores[participant]["THE QUESTION MASTER"] = 1 if question_percentage > 25 else 0

max_caring = max(
    sum(
        1
        for message in messages
        if message["sender"] == participant
        for word in caring_words
        if word in message["text"].lower()
    )
    for participant in participants
)

max_comedian = max(
    sum(
        1
        for message in messages
        if message["sender"] == participant
        for word in message["text"].lower().split()
        if word.strip(".,!?;:\"'()[]{}") in comedian_words
    )
    / message_counts[participant]
    for participant in participants
)

for participant in participants:
    caring_count = sum(
        1
        for message in messages
        if message["sender"] == participant
        for word in caring_words
        if word in message["text"].lower()
    )

    comedian_count = sum(
        1
        for message in messages
        if message["sender"] == participant
        for word in message["text"].lower().split()
        if word.strip(".,!?;:\"'()[]{}") in comedian_words
    )

    comedian_percentage = comedian_count / message_counts[participant]

    if caring_count == max_caring:
        scores[participant]["THE GROUP MOM"] = 1

    if comedian_percentage == max_comedian:
        scores[participant]["THE COMEDIAN"] = 1

priority = [
    "THE SPAMMER",
    "THE GROUP MOM",
    "THE NIGHT OWL",
    "THE STORYTELLER",
    "THE DRAMA QUEEN",
    "THE GHOST",
    "THE COMEDIAN",
    "THE QUESTION MASTER"
]

final_archetypes = {}

for participant in participants:
    highest_score = max(scores[participant].values())

    for archetype in priority:
        if scores[participant][archetype] == highest_score:
            final_archetypes[participant] = archetype
            break

print("PERSONALITY ARCHETYPES")
print()

for participant in sorted(final_archetypes):
    print(
        f"{participant:<10} : {final_archetypes[participant]}"
    )

PERSONALITY ARCHETYPES

Aman       : THE NIGHT OWL
Karan      : THE STORYTELLER
Neha       : THE DRAMA QUEEN
Priya      : THE GROUP MOM
Rahul      : THE SPAMMER
Vikas      : THE GHOST


##8. Final Report

In [20]:
print()
print("╔════════════════════════════════════════════════════╗")
print("║                 GROUPDNA FINAL REPORT              ║")
print("╚════════════════════════════════════════════════════╝")

print()
print("GROUP OVERVIEW")
print("────────────────────────────────────────────────────")
print(f"Participants        : {len(participants)}")
print(f"Total Messages      : {len(messages)}")
print(f"Media Messages      : {media_count}")
print(f"Deleted Messages    : {deleted_count}")
print(f"Chat Period         : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")

print()
print("MOST ACTIVE PERIOD")
print("────────────────────────────────────────────────────")
print(f"Most Active Day     : {most_active_day}")
print(f"Messages on Day     : {day_counts[most_active_day]}")
print(f"Most Active Hour    : {most_active_hour:02d}:00")
print(f"Messages in Hour    : {hour_counts[most_active_hour]}")

print()
print("TOP PARTICIPANTS")
print("────────────────────────────────────────────────────")

for participant in sorted(message_counts, key=message_counts.get, reverse=True):
    print(f"{participant:<10} : {message_counts[participant]} messages")

print()
print("TOP WORDS")
print("────────────────────────────────────────────────────")

for word in top_words[:10]:
    print(f"{word:<15} : {word_counts[word]}")

print()
print("RESPONSE TIME")
print("────────────────────────────────────────────────────")

for participant in sorted(response_times):
    times = response_times[participant]

    if len(times) > 0:
        average = sum(times) / len(times)
        print(f"{participant:<10} : {average:.2f} minutes")
    else:
        print(f"{participant:<10} : No responses")

print()
print("SILENT DAYS")
print("────────────────────────────────────────────────────")

for participant in sorted(silent_days):
    print(f"{participant:<10} : {len(silent_days[participant])} days")

print()
print("PERSONALITY ARCHETYPES")
print("────────────────────────────────────────────────────")

for participant in sorted(final_archetypes):
    print(f"{participant:<10} : {final_archetypes[participant]}")

print()
print("════════════════════════════════════════════════════")
print("              END OF GROUPDNA REPORT")
print("════════════════════════════════════════════════════")


╔════════════════════════════════════════════════════╗
║                 GROUPDNA FINAL REPORT              ║
╚════════════════════════════════════════════════════╝

GROUP OVERVIEW
────────────────────────────────────────────────────
Participants        : 6
Total Messages      : 3174
Media Messages      : 32
Deleted Messages    : 15
Chat Period         : 01 April 2024 to 30 May 2024

MOST ACTIVE PERIOD
────────────────────────────────────────────────────
Most Active Day     : Wednesday
Messages on Day     : 483
Most Active Hour    : 18:00
Messages in Hour    : 248

TOP PARTICIPANTS
────────────────────────────────────────────────────
Rahul      : 953 messages
Priya      : 718 messages
Neha       : 635 messages
Aman       : 490 messages
Karan      : 354 messages
Vikas      : 24 messages

TOP WORDS
────────────────────────────────────────────────────
how             : 321
guys            : 318
about           : 274
hai             : 268
am              : 260
today           : 257
his   

## Reflection

### What was the hardest part?
The challenging part was parsing the WhatsApp chat because the dataset contains system messages, media messages, deleted messages, and multiline messages, and also handling these cases correctly.

### What would I improve?
 I would add more personality archetypes and interactive filtering for participants and time periods.

### What did I learn?
This project helped me understand file handling, dictionaries, sets, loops, string processing, datetime operations, NumPy arrays, and basic data analysis using Python.

### AI Assistance
AI tools were used as a learning aid during development. The code was reviewed, tested, modified, and integrated according to the project requirements.